In [ ]:
import sys
sys.stderr = open("mcp_stderr.log", "w", encoding="utf-8", buffering=1)

# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具,差异就在3.1和3.2加ainvoke

# 3.1.定义mcp client
# MCP Client 与 MCP Server之间有两种通信协议：
# - stdio
# - streamable_http
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",# 这里是stdio通信
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },

        # 这里是http通信,本质访问网页
        "12306-mcp": {
          "transport": "streamable_http",
          "url": "https://mcp.api-inference.modelscope.net/50719920b4f24d/mcp"
        }

    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()


# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下下周四广州到北京的高铁")]}
)
print(response)

: 